In [1]:
import csv
import gzip
import json
import re
import datetime as dt
from itertools import islice
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

from ccl_science_data.common import *

loading  /home/borza/science-data/.env
OA_ROOT:  /mnt/hdd/shared/openalex-parsed
SNAP_ROOT:  /mnt/hdd/shared/openalex-snapshot-2025-03


In [2]:
ilen = len(ComC.ID_PREFIX) + 1

In [3]:
def parse_one_id(id_str):
    if id_str is None:
        return 0
    return int(id_str[ilen:])

In [4]:
min_h_index = 3
at_least_n_works = 1
in_at_least_m_years = 6
since_year = 2015

In [5]:
e_rexes = {k: re.compile(f"https://openalex.org/{k}(\d+)".encode()) for k in ["A", "W"]}

In [6]:
def get_first_id(line, pref="W"):
    return next(e_rexes[pref].finditer(line)).groups()[0].decode()

In [7]:
def filter_fun(ad):
    return (
        ad.get("summary_stats", {}).get("h_index", 0) >= min_h_index
        and sum(
            e["works_count"] >= at_least_n_works
            for e in ad["counts_by_year"]
            if e["year"] >= since_year
        )
        >= in_at_least_m_years
    )

In [9]:
tmp_path = oa_persistent / f"temp-{min_h_index}-{at_least_n_works}-{in_at_least_m_years}-{since_year}-authors.json"

In [ ]:
if tmp_path.exists():
    rel_authors = json.loads(tmp_path.read_text())
else:
    rel_authors = []
    for _, al in iter_snap_items(EntC.AUTHORS):
        ad = json.loads(al)
        if filter_fun(ad):
            rel_authors.append(parse_one_id(ad["id"]))

In [ ]:
len(rel_authors) / 1e6

In [ ]:
au_rex = re.compile(f"openalex\.org/A(\d+)".encode())

In [ ]:
rel_author_set = set(rel_authors)

In [ ]:
date_v = dt.date.today()

In [ ]:
out_dir = Path(f"/mnt/hdd/shared/chiara-openalex-trajectories-v{date_v}")
out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
class Ctx:
    def __init__(self, out_dir):
        self._files = (
            self.arecs,
            self.crecs,
            self.conrecs,
            self.toprecs,
            self.wstats,
        ) = [
            GzWriter(out_dir, name)
            for name in [
                "authorships",
                "citation-counts",
                "work-concepts",
                "work-topics",
                "work-stats",
            ]
        ]

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        for f in self._files:
            f.close(*exc)
        return False

In [ ]:
skeys = [
    "publication_year",
    "publication_date",
    "authors_count",
    "concepts_count",
    "topics_count",
]

In [ ]:
with Ctx(out_dir) as ctx:
    for _, wl in islice(iter_snap_items(EntC.WORKS), int(2 * 10e10)):
        for amatch in au_rex.finditer(wl):
            if int(amatch.groups()[0]) in rel_author_set:
                wdic = json.loads(wl)
                wid = parse_one_id(wdic["id"])
                base = {"work_id": wid}
                ctx.wstats.append(base | {k: wdic.get(k) for k in skeys})
                for ad in wdic.get("authorships", []):
                    aid = parse_one_id(ad.get("author", {}).get("id"))
                    if aid in rel_author_set:
                        ctx.arecs.append(
                            base
                            | {
                                "author_id": aid,
                                "pos": ad.get("author_position"),
                                "countries": ";".join(ad.get("countries", [])),
                            }
                        )
                for cy in wdic.get("counts_by_year", []):
                    ctx.crecs.append(base | cy)
                for k, l in [("concepts", ctx.conrecs), ("topics", ctx.toprecs)]:
                    for con in wdic.get(k, []):
                        l.append(
                            base
                            | {
                                f"{k[:-1]}_id": parse_one_id(con.get("id")),
                                "score": con.get("score"),
                            }
                        )
                break

In [ ]:
covid_concepts = [
    "https://openalex.org/concepts/C3008058167",
    "https://openalex.org/concepts/C3006700255",
    "https://openalex.org/concepts/C3007834351",
]